# Praktikum 5: Notebook 1: Spark DataFrame transformatsioonid

**Moodul 5: Suurandmed ja pilvelahendused (edasijõudnud)**

**Hinnanguline aeg:** 30 minutit

**Eeldused:** Aktiivne Databricks Free Edition konto. Sa oled läbi lugenud [praktikumi README](./README.md) ja tead, et `spark` muutuja on juba olemas. SparkSession'i käsitsi luua ei tule.

## Mida me õpime

DataFrame API on Spark'i peamine tööriist tabelikujuliste andmete töötlemisel. Selles notebook'is teed sama asja mitmel viisil. Pythoni API-ga ja SQL-iga. Ning harjutad neid transformatsioone, mida kohtad iga päev andmetorustikes.

Teemad:

1. DataFrame loomine ja uurimine
2. Veeru- ja ridade transformatsioonid (select, filter, withColumn)
3. Agregatsioonid (groupBy, agg)
4. Joinid (inner, left, anti)
5. Window functions (row_number, rank, lag, lead)
6. SQL vs DataFrame API võrdlus

Iga teema lõpus on lühike **Proovi ise** harjutus.

## 1. DataFrame loomine ja uurimine

Alustame Databricksi sisseehitatud näidistabeliga `samples.nyctaxi.trips`. See on väike (mõnikümmend tuhat rida), kiire ja alati saadaval. Suurem andmestik tuleb notebook'ides 02 ja 03.

In [ ]:
# Loeme tabeli DataFrame'iks
trips = spark.table("samples.nyctaxi.trips")

# Skeem ja esimesed read
trips.printSchema()
trips.show(5, truncate=False)

In [ ]:
# display() on Databricksi rikkalik väljund (tabeli, diagrammi toega)
display(trips.limit(10))

In [ ]:
# describe() annab kiire ülevaate numbrilistest veergudest
display(trips.describe("trip_distance", "fare_amount"))

`printSchema()` ja `describe()` on esimesed käsud, mida uue andmestiku peal jooksutada. Need ei käivita klastril raskeid päringuid, aga annavad piisavalt infot, et järgmisi samme planeerida.


Viide: [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)

## 2. Veeru- ja ridade transformatsioonid

Spark DataFrame on immutable. Iga transformatsioon tagastab uue DataFrame'i, originaal ei muutu. Sel on kaks tagajärge: turvalisus (ei saa kogemata andmeid üle kirjutada) ja laisk hindamine (Spark ehitab käskude jadast optimeeritud plaani ja käivitab alles siis, kui on vaja tulemust).

In [ ]:
import pyspark.sql.functions as F

# select + withColumn + filter ahelana
long_trips = (
    trips
    .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount")
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .where(F.col("trip_distance") > 5)
)

display(long_trips.limit(10))

In [ ]:
# withColumn saab ka olemasolevat veergu üle kirjutada
rounded = trips.withColumn("fare_amount", F.round(F.col("fare_amount"), 1))
display(rounded.select("fare_amount").limit(5))

In [ ]:
# Tingimuslik veerg: CASE WHEN ekvivalent
classified = trips.withColumn(
    "trip_class",
    F.when(F.col("trip_distance") < 1, "short")
     .when(F.col("trip_distance") < 5, "medium")
     .otherwise("long")
)

display(classified.groupBy("trip_class").count())

### Proovi ise 1

Kirjuta transformatsiooni, mis:

1. Filtreerib välja read, kus `fare_amount` on väiksem kui 2.50 (miinimum tariif).
2. Arvutab uue veeru `trip_duration_min`: sõidu kestus minutites `tpep_dropoff_datetime` ja `tpep_pickup_datetime` vahe põhjal.
3. Filtreerib välja read, kus kestus on 0 või negatiivne (andmeviga).

In [ ]:
# Sinu lahendus


## 3. Agregatsioonid

`groupBy` + `agg` on iga analüütilise päringu tuum. Kui soovid mitut agregaati korraga, kasuta `agg()` koos `F.sum`, `F.avg`, `F.max` jne funktsioonidega.


Viide: [Agregatsiooni funktsioonid](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#aggregate-functions)

In [ ]:
# Päevane kokkuvõte
daily = (
    trips
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy("pickup_date")
)

display(daily)

### Proovi ise 2

Leia top 5 päeva suurima keskmise sõidumaksu (`avg_fare`) järgi. Kasuta eelneva lahtri tulemust ja lisa lõpus `.orderBy(F.desc("avg_fare")).limit(5)`.

In [ ]:
# Sinu lahendus


## 4. Joinid

Valmistame ette kaks DataFrame'i, millega saame harjutada erinevaid join-tüüpe.

- `trips_enriched`: taksosõidud päeva ja tunni tasemel
- `days`: sünteetiline kalender 2016 aasta kohta (pühad)

Kalendri teeme käsitsi, et näidata Pythoni list → DataFrame konversiooni.

In [ ]:
from datetime import date

# Väike puhkepäevade tabel
holidays = spark.createDataFrame(
    [
        (date(2016, 1, 1),  "New Year"),
        (date(2016, 1, 18), "MLK Day"),
        (date(2016, 2, 15), "Presidents Day"),
    ],
    schema="holiday_date DATE, holiday_name STRING",
)

trips_with_date = trips.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

display(holidays)

In [ ]:
# INNER JOIN: ainult sõidud, mis toimusid puhkepäeval
holiday_trips = trips_with_date.join(
    holidays,
    trips_with_date.pickup_date == holidays.holiday_date,
    "inner",
)

print("Puhkepäevade sõidud:", holiday_trips.count())
display(holiday_trips.select("pickup_date", "holiday_name", "fare_amount").limit(5))

In [ ]:
# LEFT JOIN: kõik sõidud, puhkepäeva nimi kui olemas
trips_flagged = trips_with_date.join(
    holidays,
    trips_with_date.pickup_date == holidays.holiday_date,
    "left",
)

display(
    trips_flagged
    .groupBy(F.col("holiday_name").isNotNull().alias("is_holiday"))
    .agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
)

In [ ]:
# LEFT ANTI JOIN: sõidud, mis EI toimunud puhkepäeval
# Tüüpiline kasutus: leida read, mis ei kuulu teise tabelisse
non_holiday = trips_with_date.join(
    holidays,
    trips_with_date.pickup_date == holidays.holiday_date,
    "left_anti",
)

print("Mitte-puhkepäevade sõidud:", non_holiday.count())

**Miks left anti join?** Sama tulemuse saaks `left join` + `where holiday_name is null` kaudu, aga anti join on optimeerijale selgem ja jõuab harva tõsiseid andmeid ridadest välja lugeda. See on väike, aga jõudluse mõttes reaalne vahe suurtel andmetel.


Viide: [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)

### Proovi ise 3

Loo uus `weekdays` DataFrame, kus on kõik nädalapäevad (1-7) ja nende nimed (näiteks "Monday", "Tuesday" jne). Tee sellega inner join `trips_with_date` peale, et igale sõidule lisada nädalapäeva nimi. Vihje: `F.dayofweek("pickup_date")` annab numbri 1-7.

In [ ]:
# Sinu lahendus


## 5. Window functions

Window function'id arvutavad väärtusi üle "akna". Mingi seotud ridade hulga, ilma et sa kaotaks detailset taset. Tüüpilised kasutused:

- järjestamine grupi sees (`row_number`, `rank`, `dense_rank`)
- eelmise / järgmise rea vaatamine (`lag`, `lead`)
- jooksev summa või keskmine (`sum`, `avg` üle window)

Window koosneb kahest osast: `partitionBy` (kuidas jagada) ja `orderBy` (kuidas järjestada).


Viide: [Window functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)

In [ ]:
from pyspark.sql.window import Window

# Väiksem andmestik, et Databricks display oleks kiire
sample = (
    trips
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .where(F.col("pickup_date").between("2016-01-01", "2016-01-07"))
    .select("pickup_date", "tpep_pickup_datetime", "fare_amount")
)

# Järjesta iga päeva sees sõidumaksu järgi (kalleim = 1)
w_fare = Window.partitionBy("pickup_date").orderBy(F.col("fare_amount").desc())

ranked = (
    sample
    .withColumn("fare_rank", F.row_number().over(w_fare))
    .where(F.col("fare_rank") <= 3)
    .orderBy("pickup_date", "fare_rank")
)

display(ranked)

In [ ]:
# lag: eelmise rea väärtus sama partitsiooni sees
w_time = Window.partitionBy("pickup_date").orderBy("tpep_pickup_datetime")

with_prev = (
    sample
    .withColumn("prev_fare", F.lag("fare_amount").over(w_time))
    .withColumn("fare_delta", F.col("fare_amount") - F.col("prev_fare"))
)

display(with_prev.limit(20))

### Proovi ise 4

Kasuta window function'it, et leida iga päeva esimene ja viimane sõit (`tpep_pickup_datetime` järgi). Vihje: `row_number()` üle päeva, kasvavalt + kahanevalt, või `first_value` / `last_value` üle kogu akna.

In [ ]:
# Sinu lahendus


## 6. SQL vs DataFrame API

Spark SQL ja DataFrame API tekitavad täpselt sama täidetava plaani. Kumb valida?

- **SQL** on loetavam siis, kui päring on puhas SELECT / GROUP BY / JOIN ilma keerulise programmiloogika järele. Analüütikud saavad lugeda.
- **DataFrame API** on mugavam siis, kui kirjutad üldise transformatsiooni, mida taaskasutad, või kui tahad Pythoni tingimuslikku ülesehitust.

Mõlemat saab segada. SQL tabelite nimed tuleb SQL kontekstis registreerida. Selleks `createOrReplaceTempView`.


Viide: [Spark SQL funktsioonid](https://spark.apache.org/docs/latest/api/sql/index.html)

In [ ]:
# Registreerime ajutise vaate
trips.createOrReplaceTempView("trips_view")

# Sama päring SQL-is
daily_sql = spark.sql("""
    SELECT
        CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(SUM(fare_amount), 2) AS total_fare
    FROM trips_view
    GROUP BY CAST(tpep_pickup_datetime AS DATE)
    ORDER BY pickup_date
""")

display(daily_sql)

In [ ]:
# Veendume, et kaks varianti annavad sama tulemuse
print("DataFrame API ridu:", daily.count())
print("SQL ridu:           ", daily_sql.count())

In [ ]:
# Võrdleme täitmisplaane: DataFrame API vs SQL
print("=== DataFrame API ===")
daily.explain("formatted")

print("\n=== SQL ===")
daily_sql.explain("formatted")

### Proovi ise 5

Võta Proovi ise 3 lahendus (weekdays join) ja kirjuta sama loogika puhtalt SQL-is, kasutades `spark.sql(...)`. Kumb variant on sinu arvates loetavam?

In [ ]:
# Sinu lahendus


## Kokkuvõte

Selles notebook'is sa:

- Lugesid Databricksi näidistabelit ja uurisid skeemi
- Harjutasid `select`, `withColumn`, `filter`, `groupBy`, `agg` ahelaid
- Tegid inner, left ja left anti joine ning said aru, miks viimast võib eelistada
- Kasutasid window function'eid järjestamiseks ja `lag` operatsiooniks
- Nägid, et SQL ja DataFrame API annavad sama tulemuse

**Mida meeles pidada:**

- Spark DataFrame on immutable. Iga transformatsioon on uus DataFrame.
- Transformatsioonid on laisk: päring käivitub siis, kui kutsud action'i (`count`, `show`, `display`, `collect`, `write`).
- `explain()` näitab, kuidas Spark sinu päringut täidab. Iga optimeerimisotsus algab sellest.

Järgmine notebook (02_failiformaadid_ja_partitsioneerimine.ipynb) võtab ette täismahus NYC Taxi andmestiku ja mõõdab, kuidas formaadivalik ja partitsioneerimine jõudlust mõjutavad.